In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
from config import *
import os
import re
import pandas as pd
import numpy as np
import glob
import pandas as pd
# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Do not truncate column values
pd.set_option('display.max_colwidth', None)

In [2]:
base_dir = 'data/preprocessed data'
pattern  = os.path.join(base_dir, '_*y.csv')
file_list = sorted(glob.glob(pattern))

all_y = None
for fp in file_list:
    df = pd.read_csv(fp)
    # ensure the composite key exists
    if 'hhidpn' not in df.columns:
        df['hhidpn'] = df['HHID'].astype(str) + df['PN'].astype(str)
    # drop the original key columns
    df = df.drop(['HHID', 'PN'], axis=1, errors='ignore')
    # merge on hhidpn only
    if all_y is None:
        all_y = df
    else:
        all_y = all_y.merge(df, on='hhidpn', how='outer')

# write out, using "NA" for any missing values
out_path = os.path.join(base_dir, 'all_y.csv')
all_y.to_csv(out_path, index=False, na_rep='NA')

all_y.head()


,hhidpn,00demcls,02demcls,04demcls,06demcls,08demcls,10demcls,12demcls,14demcls,16demcls,18demcls,20demcls
0,1010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
3,3020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
4,10001010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


**Incident-dementia outcomes**

Estimand: among participants aged > 50 and assessed dementia-free at baseline, the probability that dementia is observed at a biennial follow-up assessment within the window (6 or 8 years), among those whose status is known at the end of the window.

- `y = 1`: dementia (`demcls == 1`) at any follow-up wave of the window.
- `y = 0`: no dementia at any follow-up wave **and** assessed dementia-free at the final wave of the window.
- `y = NA` (excluded): not assessed at the final wave without observed dementia, i.e. died, lost to follow-up, or did not respond.

The baseline wave is not part of the outcome (baseline eligibility is applied in the data-management notebooks).

Sensitivity outcome (`*_any`): `y = 0` if assessed dementia-free at *any* follow-up wave, with no dementia observed.

In [3]:
WINDOWS = {
    'demcls00-06': ['02', '04', '06'],
    'demcls00-08': ['02', '04', '06', '08'],
    'demcls10-16': ['12', '14', '16'],
    'demcls10-18': ['12', '14', '16', '18'],
}

def incident_outcome(df, follow_up, require_final_wave=True):
    fu = df[[f'{w}demcls' for w in follow_up]]
    incident = (fu == 1).any(axis=1)
    if require_final_wave:
        assessed_free = ~incident & (fu.iloc[:, -1] == 0)
    else:
        assessed_free = ~incident & (fu == 0).any(axis=1)
    return pd.Series(np.where(incident, 1.0, np.where(assessed_free, 0.0, np.nan)), index=df.index)

outcome_cols = []
for name, follow_up in WINDOWS.items():
    all_y[name] = incident_outcome(all_y, follow_up)
    all_y[f'{name}_any'] = incident_outcome(all_y, follow_up, require_final_wave=False)
    outcome_cols += [name, f'{name}_any']
    print(name, all_y[name].value_counts(dropna=False).to_dict())

demcls00-06 {nan: 22991, 0.0: 16705, 1.0: 2709}
demcls00-08 {nan: 23741, 0.0: 15404, 1.0: 3260}
demcls10-16 {nan: 20401, 0.0: 19150, 1.0: 2854}
demcls10-18 {nan: 23518, 0.0: 15636, 1.0: 3251}


In [4]:
# Outcome status in the eligible baseline cohorts (Reviewer 1, comment 1a): incident dementia,
# assessed dementia-free, died before the final wave without observed dementia, and alive (or vital status
# unknown) but not assessed at the final wave. Vital status at the final wave is from the HRS tracker
# (xALIVE: 1 alive, 2 presumed alive, 5 deceased this wave, 6 deceased as of a prior wave).
FINAL_WAVE_VITAL = {'demcls00-06': 'KALIVE', 'demcls00-08': 'LALIVE', 'demcls10-16': 'PALIVE', 'demcls10-18': 'QALIVE'}
tracker = pd.read_stata('data/original data/trk2022tr_r.dta', columns=['HHID', 'PN'] + list(FINAL_WAVE_VITAL.values()),
                        convert_categoricals=False)
tracker['hhidpn'] = (tracker['HHID'] + tracker['PN']).astype(int)

STATUS_ORDER = ['incident dementia', 'assessed dementia-free',
                'died before final wave, no dementia observed', 'alive or unknown, not assessed at final wave']
status_rows = []
for name, follow_up in WINDOWS.items():
    baseline = pd.read_csv(os.path.join(base_dir, f'20{name[6:8]}.csv'), usecols=['hhidpn', 'agey_m', 'demcls'])
    eligible = baseline[(baseline['agey_m'] > 50) & (baseline['demcls'] == 0)]
    d = (eligible[['hhidpn']].merge(all_y[['hhidpn', name]], on='hhidpn', how='left')
         .merge(tracker[['hhidpn', FINAL_WAVE_VITAL[name]]], on='hhidpn', how='left'))
    died = d[FINAL_WAVE_VITAL[name]].isin([5, 6])
    status = pd.Series(np.select([d[name] == 1, d[name] == 0, died], STATUS_ORDER[:3], STATUS_ORDER[3]))
    counts = status.value_counts().reindex(STATUS_ORDER, fill_value=0)
    status_rows.append({'window': name, 'eligible': len(d), **counts.to_dict()})

outcome_status = pd.DataFrame(status_rows)
outcome_status.to_csv(os.path.join(base_dir, 'outcome_status.csv'), index=False)
outcome_status

,window,eligible,incident dementia,assessed dementia-free,"died before final wave, no dementia observed","alive or unknown, not assessed at final wave"
0,demcls00-06,17336,1593,12065,2467,1211
1,demcls00-08,17336,2015,10927,3167,1227
2,demcls10-16,19243,1699,12960,2398,2186
3,demcls10-18,19243,2000,10564,3112,3567


In [5]:
wave_cols = [c for c in all_y.columns if c[:2].isdigit() and c.endswith('demcls')]
all_y = all_y[['hhidpn'] + wave_cols + outcome_cols]
all_y.to_csv(os.path.join(base_dir, 'all_y.csv'), na_rep='NA', index=False)